# Archived experiment notebook

Source preserved for review. Private outputs, attachments, and execution counts are removed in this public copy; original AWS notebooks remain untouched. This copy is not evidence of a fresh execution. Use the public Research Review for aggregate results. Private input contracts and artifacts are required to reproduce this historical experiment.


# NFL interaction-feature review
**Feature engineering remains open. This notebook does not train or score a new model.**

Review the output of the bounded **32-training-play** smoke test in the current canonical repository. It exposes how much synchronized relative-motion information is actually observed, which channels are missing, and whether gaps remain gaps. It does not establish a feature's RMSE contribution.

First run `inspect-result`, then `feature-smoke` as described in `START_HERE.md`. The stored-run inspection is a separate existing experiment; it must not be labeled a temporal-edge feature gain.


In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown
pio.renderers.default = 'plotly_mimetype'
EVIDENCE = Path(os.environ.get('NFL_NOTEBOOK_EVIDENCE_ROOT', str(Path.home() / 'nfl-workspace-evidence')))
FIGURES = []
def receipt(name):
    path = EVIDENCE / ('latest_' + name + '.json')
    return json.loads(path.read_text()) if path.is_file() else {}
def present(fig):
    FIGURES.append(fig)
    fig.show()
def offline_report(filename, title):
    destination = EVIDENCE / 'notebook_views' / filename
    destination.parent.mkdir(parents=True, exist_ok=True)
    body = '<!doctype html><html><head><meta charset="utf-8"><title>'+title+'</title></head><body><h1>'+title+'</h1>'
    for i, figure in enumerate(FIGURES):
        body += figure.to_html(full_html=False, include_plotlyjs=True if i == 0 else False)
    body += '</body></html>'
    destination.write_text(body)
    print('Saved offline, private HTML:', destination)
    return str(destination)


In [ ]:
report=receipt('feature-smoke')
inspection=receipt('inspect-result')
fixture=bool(report.get('_synthetic_fixture'))
if fixture:
    display(Markdown('## SYNTHETIC SOFTWARE-TEST FIXTURE — NOT NFL EXPERIMENT EVIDENCE'))
if not report:
    print('No feature-smoke receipt. Run the terminal smoke command first; no substitute data will be plotted.')
else:
    display(pd.DataFrame([{k:report.get(k) for k in ['status','plays','valid_pair_frames','max_tensor_bytes','validation_scored','scientific_fits','new_rmse']}]))
if inspection.get('recomputed'):
    display(Markdown('### Separate, previously completed coordinate-versus-velocity experiment'))
    values=inspection['recomputed']
    display(pd.DataFrame([{k:values.get(k) for k in ['rows','games','control_rmse','velocity_rmse','relative_gain']}]))
    print('These are saved-error recomputations, not a new feature experiment or a numerical forward replay from weights.')


## Channel support over the 32-play smoke
The denominator is valid jointly observed pair-frames. Zero-valued measurements and missing measurements have different meanings. Low support triggers a missingness/clock audit, not filling gaps with invented values.


In [ ]:
names=report.get('channel_names',[])
if names and report.get('valid_pair_frames',0)>0:
    coverage=pd.DataFrame({'Channel':names,'Valid pair-frames':report['channel_valid_counts']})
    coverage['Support percent']=100*coverage['Valid pair-frames']/report['valid_pair_frames']
    display(coverage)
    fig=px.bar(coverage,x='Support percent',y='Channel',orientation='h',
               title='Training-only channel support across 32 plays')
    fig.update_layout(height=490)
    present(fig)


## First training play: observed support and gaps
Only the first play's feature tensors are saved by the canonical smoke. Axes below show **observed tensor indices**, not invented frame IDs or elapsed seconds. The view is descriptive, not a representative validation sample. No player identity or true defensive assignment is inferred from a tensor index.


In [ ]:
example = Path(report.get('output',''))/'edges_example.npz' if report.get('output') else None
arrays=None
if example and example.is_file():
    import hashlib
    expected=report.get('example_sha256')
    actual=hashlib.sha256(example.read_bytes()).hexdigest()
    if not expected or expected != actual:
        raise ValueError('Example tensor checksum failed; no data are plotted')
    with np.load(example,allow_pickle=False) as stored:
        arrays={k:stored[k] for k in ('values','valid','pair_valid','adjacent')}
    values, valid, pairs = arrays['values'], arrays['valid'], arrays['pair_valid']
    if values.ndim!=4 or valid.shape!=values.shape or pairs.shape!=values.shape[:3] or values.shape[-1]!=len(names):
        raise ValueError('Unexpected tensor layout')
    fractions=valid.sum(axis=(0,1)).T/np.maximum(pairs.sum(axis=(0,1)),1)[None,:]
    fig=go.Figure(go.Heatmap(z=fractions,x=list(range(values.shape[2])),y=names,
        zmin=0,zmax=1,colorbar={'title':'Support fraction'}))
    fig.update_layout(title='First training play: channel support at each observed tensor index',
                      xaxis_title='Observed tensor index',height=510)
    present(fig)
else:
    print('No verified private example tensor is available. No synthetic example is substituted.')


In [ ]:
if arrays is not None:
    pair_count=pairs.sum(axis=2)
    fig=go.Figure(go.Heatmap(z=pair_count,x=list(range(pair_count.shape[1])),
        y=list(range(pair_count.shape[0])),colorbar={'title':'Joint frames'}))
    fig.update_layout(title='First training play: simultaneous observation support by player pair',
        xaxis_title='Destination-player index',yaxis_title='Source-player index',height=480)
    present(fig)


In [ ]:
if arrays is not None:
    channel_index=0  # Change only to inspect another named input channel; this is not model tuning.
    pair_support=valid[:,:,:,channel_index].sum(axis=2)
    if np.max(pair_support)>0:
        source,destination=np.unravel_index(np.argmax(pair_support),pair_support.shape)
        series=np.where(valid[source,destination,:,channel_index],values[source,destination,:,channel_index],np.nan)
        fig=go.Figure(go.Scatter(x=list(range(len(series))),y=series,mode='lines+markers',connectgaps=False))
        fig.update_layout(title=f'First training play: {names[channel_index]} for pair {source} → {destination}',
            xaxis_title='Observed tensor index',yaxis_title='Stored feature units / scale',height=390)
        present(fig)
        print('Pair chosen by observed channel support only; missing frames remain gaps. Consult canonical NAMES/scales for physical units.')


## Next scientific gate — not executed here
After exact input support, cache lineage, and existing-run recovery are verified, compare actual synchronized pair histories with a capacity-matched terminal-information control. Keep the same architecture, training exposure, seed, normalization, split, loss, labels, and forecast rows. Fit transforms and screen candidate channels using the training partition only.

Predeclare the missing-derivative policy and time encoding in the control: neither stale observations nor fake zero derivatives should masquerade as information. Confirm raw-input adapter parity and fresh-process checkpoint recovery before one bounded chronological fold. A proposed continuation rule is at least **1% lower official coordinate RMSE** and an upper paired-game 95% difference bound below zero; then replicate across later folds/seeds. This is not a guarantee of 0.46340 or a license to tune on the already inspected final holdout.

No learned temporal-edge encoder comparison is implemented by this readiness kit. The canonical feature prototype exists; its scientific value remains unmeasured. See `FEATURE_RESEARCH_PLAN.md` and repository `docs/TEMPORAL_EDGE_PROTOCOL.md`.


In [ ]:
if FIGURES:
    html_path=offline_report('interaction_feature_review.html','NFL observed interaction-feature review' + (' — SYNTHETIC TEST FIXTURE' if fixture else ''))
print('Model fits: 0. New feature RMSE: none. Feature research: OPEN.')
